# VIX Deep Learning Pipeline — Version enrichie

Notebook complet intégrant :

- correction robuste des valeurs `NaN` et `inf` dans les blocs Kalman, HMM, Heston et VRP ;
- métriques complètes : accuracy, precision, recall, F1 et AUC ;
- feature engineering des interactions sur toutes les features candidates, avec garde-fou mémoire ;
- sélection finale après engineering global ;
- graphes de visualisation des résultats ;
- export Excel enrichi ;
- suggestions méthodologiques à la fin.

> Remarque : le notebook est volontairement défensif. Les blocs les plus coûteux, comme les interactions globales, sont contrôlés par `CONFIG` afin d'éviter une explosion mémoire.


In [1]:
# ============================================================
# 0. Installation et imports
# ============================================================

import sys
!{sys.executable} -m pip install -q arch pykalman hmmlearn shap xgboost xlsxwriter imbalanced-learn yfinance pandas_datareader statsmodels

import os
import time
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import yfinance as yf
import pandas_datareader.data as web

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from imblearn.combine import SMOTETomek

import matplotlib.pyplot as plt
import seaborn as sns

import shap
from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 5.3 MB/s eta 0:00:00
Device : cpu


In [2]:
# ============================================================
# 1. Configuration centrale
# ============================================================

CONFIG = {
    'seed': 42,
    'start_date': '2012-01-01',
    'lookback': 21,
    'horizons': [1, 3, 5, 7],
    'flat_thr': 0.003,

    # Sélection finale après engineering global.
    'top_n_final': 80,

    # Garde-fous mémoire pour l'engineering sur toutes les features.
    # None = garde toutes les interactions retenues par batch, mais peut être très lourd.
    'max_interaction_keep': 3000,
    'interaction_batch_size': 1500,
    'interaction_keep_per_batch': 500,

    'batch_size': 64,
    'epochs': 60,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'dropout': 0.3,
    'class_quantiles': [0.25, 0.75],
    'class_labels': ['DOWN_FORT', 'DOWN_FAIBLE', 'UP_FAIBLE', 'UP_FORT'],
}

YF_TICKERS = """
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX ^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD AAPL AMZN MSFT NVDA INTC QCOM CSCO XOM WMT MCD
SBUX MS COF BLK SCHW CLX CPB GIS BMY CI BDX LMT NOC GD HON DE ITT CCI AVB PSA
EQIX EXC NEE TXN LOGI VOD ENB PAYX LUV CMCSA
""".split()

FRED_SERIES = {
    'NFCI': 'NFCI',
    'STLFSI': 'STLFSI4',
    'T10Y2Y': 'T10Y2Y',
    'EFFR': 'EFFR',
    'VXVCLS': 'VXVCLS',
}

TARGET_COL = 'VIX_Amplitude_Class'
print("Configuration chargée.")


Configuration chargée.


In [3]:
# ============================================================
# 2. Chargement des données
# ============================================================

def load_data(start=CONFIG['start_date']):
    t0 = time.time()

    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^', 'IDX_').replace('-', '_') for c in raw.columns]

    coverage = raw.notna().mean()
    raw = raw.loc[:, coverage >= 0.90]
    raw = raw.ffill().dropna(how='all')

    fred_frames = []
    for name, series_id in FRED_SERIES.items():
        try:
            s = web.DataReader(series_id, 'fred', start).squeeze()
            s.name = f'FRED_{name}'
            fred_frames.append(s)
        except Exception as e:
            print(f"  [WARN] FRED {series_id}: {e}")

    if fred_frames:
        fred_df = pd.concat(fred_frames, axis=1).reindex(raw.index, method='ffill')
        raw = pd.concat([raw, fred_df], axis=1)

    raw = raw.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    print(f"  Dataset : {raw.shape[0]} jours x {raw.shape[1]} séries ({time.time() - t0:.1f}s)")
    return raw

df_raw = load_data()


  Dataset : 3784 jours x 64 séries (29.2s)


In [4]:
# ============================================================
# 3. Features Time-Series : EGARCH, Kalman, HMM, Heston, VRP, Jumps, Hawkes
# ============================================================

def build_ts_features(df_raw, train_end_idx):
    """
    Construit les features time-series.
    Tout fit est fait sur le train uniquement, puis appliqué au dataset complet.
    Version robuste contre NaN et inf.
    """
    t0 = time.time()
    feats = pd.DataFrame(index=df_raw.index)

    vix_col = 'IDX_VIX' if 'IDX_VIX' in df_raw.columns else [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c][0]
    spx_col = [c for c in df_raw.columns if 'GSPC' in c or 'SPY' in c][0]

    vix = df_raw[vix_col].replace([np.inf, -np.inf], np.nan).astype(float).ffill().bfill()
    spx = df_raw[spx_col].replace([np.inf, -np.inf], np.nan).astype(float).ffill().bfill()

    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)
    spx_ret = np.log(spx / spx.shift(1)).replace([np.inf, -np.inf], np.nan).fillna(0)

    # EGARCH
    try:
        spx_ret_train = spx_ret.iloc[:train_end_idx].replace([np.inf, -np.inf], np.nan).fillna(0) * 100
        am = arch_model(spx_ret_train, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
        res_eg = am.fit(disp='off', show_warning=False)
        eg_full = res_eg.forecast(start=0, reindex=True)
        condvar = eg_full.variance.iloc[:, 0] / 10000
        condvar = condvar.reindex(df_raw.index, method='ffill').replace([np.inf, -np.inf], np.nan).ffill().bfill()
        feats['egarch_condvar'] = condvar
        feats['egarch_delta'] = condvar.diff()
        print(f"  EGARCH fit OK ({time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] EGARCH: {e}")
        feats['egarch_condvar'] = np.nan
        feats['egarch_delta'] = np.nan

    # Kalman
    try:
        vix_clean = vix.replace([np.inf, -np.inf], np.nan).interpolate('linear').ffill().bfill().astype(float)
        if not np.isfinite(vix_clean.values).all():
            finite_mean = np.nanmean(vix_clean.replace([np.inf, -np.inf], np.nan).values)
            vix_clean = vix_clean.replace([np.inf, -np.inf], np.nan).fillna(finite_mean)

        train_vix_clean = vix_clean.iloc[:train_end_idx].values.reshape(-1, 1)
        full_vix_clean = vix_clean.values.reshape(-1, 1)

        kf = KalmanFilter(
            transition_matrices=np.array([[1.0]]),
            observation_matrices=np.array([[1.0]]),
            initial_state_mean=np.array([float(vix_clean.iloc[0])]),
            initial_state_covariance=np.array([[1.0]]),
            transition_covariance=np.array([[1e-3]]),
            observation_covariance=np.array([[1e-2]]),
            em_vars=['transition_covariance', 'observation_covariance']
        )
        kf = kf.em(train_vix_clean, n_iter=20)
        sm, _ = kf.filter(full_vix_clean)
        ss, _ = kf.smooth(full_vix_clean)

        kalman_filtered = pd.Series(sm[:, 0], index=df_raw.index)
        kalman_smooth = pd.Series(ss[:, 0], index=df_raw.index)
        feats['kalman_residual'] = (vix_clean - kalman_filtered).shift(1).replace([np.inf, -np.inf], np.nan)
        feats['kalman_innovation'] = (vix_clean - kalman_smooth.shift(1)).shift(1).replace([np.inf, -np.inf], np.nan)
        print(f"  Kalman fit OK ({time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] Kalman: {e}")
        feats['kalman_residual'] = np.nan
        feats['kalman_innovation'] = np.nan

    # HMM
    try:
        rv5 = vix_ret.pow(2).rolling(5, min_periods=3).mean()
        mu = vix.iloc[:train_end_idx].mean()
        sd = vix.iloc[:train_end_idx].std()
        if not np.isfinite(sd) or sd <= 1e-12:
            sd = 1.0
        vix_n = (vix - mu) / sd
        X_hmm = pd.DataFrame({'ret': vix_ret, 'vol5': np.sqrt(rv5), 'level': vix_n}, index=df_raw.index)
        X_hmm = X_hmm.replace([np.inf, -np.inf], np.nan).dropna()
        train_dates = df_raw.index[:train_end_idx]
        X_tr_df = X_hmm.loc[X_hmm.index.isin(train_dates)]
        if len(X_tr_df) < 50:
            raise ValueError('Pas assez de données propres pour HMM.')

        model_hmm = hmmlib.GaussianHMM(n_components=2, covariance_type='full', n_iter=200, random_state=SEED)
        model_hmm.fit(X_tr_df.values)
        states_tr = model_hmm.predict(X_tr_df.values)
        rv5_train = rv5.reindex(X_tr_df.index)
        state_vols = []
        for s in range(2):
            vals = rv5_train.values[states_tr == s]
            vals = vals[np.isfinite(vals)]
            state_vols.append(np.nanmean(vals) if len(vals) else -np.inf)
        stress_st = int(np.argmax(state_vols))
        proba_full = model_hmm.predict_proba(X_hmm.values)
        states_full = model_hmm.predict(X_hmm.values)
        feats['hmm_p_stress'] = pd.Series(proba_full[:, stress_st], index=X_hmm.index).reindex(df_raw.index)
        feats['hmm_state'] = pd.Series(states_full, index=X_hmm.index).reindex(df_raw.index)
        print(f"  HMM fit OK ({time.time() - t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] HMM: {e}")
        feats['hmm_p_stress'] = np.nan
        feats['hmm_state'] = np.nan

    # Heston proxies
    v0 = (vix / 100).pow(2)
    theta = vix_ret.pow(2).rolling(60, min_periods=30).mean()
    vvix_col = [c for c in df_raw.columns if 'VVIX' in c]
    if vvix_col:
        xi = df_raw[vvix_col[0]].replace([np.inf, -np.inf], np.nan).astype(float).ffill().bfill() / 100
    else:
        xi = vix_ret.rolling(20, min_periods=10).std()
    rho = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    def rolling_kappa(series, w=252):
        kappa = pd.Series(np.nan, index=series.index)
        s = series.replace([np.inf, -np.inf], np.nan).ffill().bfill()
        for i in range(w, len(s)):
            y = s.iloc[i-w+1:i+1].values
            x = s.iloc[i-w:i].values
            try:
                if not np.isfinite(x).all() or not np.isfinite(y).all():
                    continue
                beta = np.corrcoef(x, y)[0, 1]
                if np.isfinite(beta) and 0 < abs(beta) < 0.9999:
                    hl = -np.log(2) / np.log(abs(beta))
                    if np.isfinite(hl) and hl > 0:
                        kappa.iloc[i] = np.log(2) / hl
            except Exception:
                pass
        return kappa.replace([np.inf, -np.inf], np.nan)

    kappa = rolling_kappa(vix)
    feats['heston_v0'] = v0
    feats['heston_theta'] = theta
    feats['heston_xi'] = xi
    feats['heston_rho'] = rho
    feats['heston_kappa'] = kappa
    feats['heston_feller'] = (2 * kappa * theta) / xi.pow(2).replace(0, np.nan)
    feats['heston_v0_minus_theta'] = v0 - theta

    for h in CONFIG['horizons']:
        ev = theta + (v0 - theta) * np.exp(-kappa * h)
        ev = ev.replace([np.inf, -np.inf], np.nan)
        feats[f'heston_ev_h{h}'] = ev
        feats[f'heston_spread_h{h}'] = v0 - ev
        feats[f'heston_vol_h{h}'] = np.sqrt(ev.clip(lower=0)) * 100

    # VRP HAR-RV
    try:
        import statsmodels.api as sm
        rv1d = vix_ret.pow(2)
        rv5d = rv1d.rolling(5, min_periods=3).mean()
        rv22d = rv1d.rolling(22, min_periods=10).mean()
        rv_target = rv1d.shift(-22).rolling(22, min_periods=11).mean()
        har_df = pd.DataFrame({'rv1': rv1d, 'rv5': rv5d, 'rv22': rv22d, 'y': rv_target}, index=df_raw.index)
        har_df = har_df.replace([np.inf, -np.inf], np.nan).dropna()
        train_dates = df_raw.index[:train_end_idx]
        har_train = har_df.loc[har_df.index.isin(train_dates)]
        if len(har_train) < 50:
            raise ValueError('Pas assez de données propres pour HAR-RV.')
        Xh = sm.add_constant(har_train[['rv1', 'rv5', 'rv22']], has_constant='add')
        har_model = sm.OLS(har_train['y'], Xh).fit()
        Xfull = sm.add_constant(har_df[['rv1', 'rv5', 'rv22']], has_constant='add')
        rv_pred = har_model.predict(Xfull).reindex(df_raw.index)
        feats['VRP'] = (vix / 100).pow(2) - rv_pred
        vrp_train = feats['VRP'].iloc[:train_end_idx]
        vrp_sd = vrp_train.std()
        if not np.isfinite(vrp_sd) or vrp_sd <= 1e-12:
            vrp_sd = 1.0
        feats['VRP_zscore'] = (feats['VRP'] - vrp_train.mean()) / vrp_sd
    except Exception as e:
        print(f"  [WARN] VRP: {e}")
        feats['VRP'] = np.nan
        feats['VRP_zscore'] = np.nan

    # Jumps et Hawkes simplifié
    sigma_60 = vix_ret.rolling(60, min_periods=30).std()
    is_jump = (vix_ret.abs() > 3 * sigma_60).astype(float)
    feats['jump_intensity_20d'] = is_jump.rolling(20, min_periods=10).mean()
    feats['jump_intensity_60d'] = is_jump.rolling(60, min_periods=30).mean()

    sigma_hw = vix_ret.rolling(30, min_periods=15).std()
    jump_times = vix_ret.index[vix_ret.abs() > 2 * sigma_hw]
    hawkes = pd.Series(0.0, index=vix_ret.index)
    for i, t in enumerate(vix_ret.index):
        past = jump_times[jump_times < t]
        if len(past) > 0:
            days_since = np.array([(t - tj).days for tj in past])
            hawkes.iloc[i] = 0.3 + 0.3 * np.sum(np.exp(-0.1 * days_since))
        else:
            hawkes.iloc[i] = 0.3
    feats['hawkes_intensity'] = hawkes
    sd_hw = hawkes.iloc[:train_end_idx].std()
    if not np.isfinite(sd_hw) or sd_hw <= 1e-12:
        sd_hw = 1.0
    feats['hawkes_zscore'] = (hawkes - hawkes.iloc[:train_end_idx].mean()) / sd_hw

    feats = feats.replace([np.inf, -np.inf], np.nan)
    print(f"  TS features OK ({time.time() - t0:.1f}s)")
    return feats


In [5]:
# ============================================================
# 4. Cible d'amplitude
# ============================================================

def build_amplitude_target(vix_series, horizon, train_end_idx, regime_series=None):
    vix = vix_series.replace([np.inf, -np.inf], np.nan).ffill().bfill().copy()
    ret = (vix.shift(-horizon) / vix) - 1
    ret = ret.replace([np.inf, -np.inf], np.nan).dropna()

    flat_mask = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat_mask]

    vix_train = vix.iloc[:train_end_idx]
    calm_thr = vix_train.quantile(0.33)
    stress_thr = vix_train.quantile(0.67)

    regime = pd.Series('NORMAL', index=ret.index)
    regime[vix.reindex(ret.index) < calm_thr] = 'CALM'
    regime[vix.reindex(ret.index) >= stress_thr] = 'STRESS'

    ret_train = ret.iloc[:train_end_idx]
    thresholds = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_train[regime.iloc[:train_end_idx] == reg]
        if len(sub) >= 20:
            thresholds[reg] = (sub.quantile(0.25), sub.quantile(0.75))
        else:
            thresholds[reg] = (ret_train.quantile(0.25), ret_train.quantile(0.75))

    def classify(r, reg):
        q25, q75 = thresholds.get(reg, (0, 0))
        if r < q25:
            return 0
        if r < 0:
            return 1
        if r < q75:
            return 2
        return 3

    target = pd.Series([classify(r, regime[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    print(f"  [Target h={horizon}j] {len(target)} obs, dist: {target.value_counts().to_dict()}")
    return target, regime, ret, thresholds


In [6]:
# ============================================================
# 5. Feature engineering avancé
# ============================================================

def build_advanced_features(df_raw, vix_series, spx_series, train_end_idx):
    feats = pd.DataFrame(index=df_raw.index)
    vix = vix_series.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    spx = spx_series.replace([np.inf, -np.inf], np.nan).ffill().bfill()
    vix_ret = np.log(vix / vix.shift(1)).replace([np.inf, -np.inf], np.nan)
    spx_ret = np.log(spx / spx.shift(1)).replace([np.inf, -np.inf], np.nan)

    feats['vix_vol_of_vol_5d'] = vix_ret.rolling(5, min_periods=3).std()
    feats['vix_vol_of_vol_10d'] = vix_ret.rolling(10, min_periods=5).std()
    feats['vix_momentum_2d'] = vix.pct_change(2)
    feats['vix_momentum_3d'] = vix.pct_change(3)
    feats['vix_acceleration'] = vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio'] = vix_ret.abs().rolling(5, min_periods=3).max() / vix_ret.abs().rolling(5, min_periods=3).mean().replace(0, np.nan)
    feats['vix_vol_ratio_5_60'] = vix_ret.rolling(5, min_periods=3).std() / vix_ret.rolling(60, min_periods=30).std().replace(0, np.nan)
    feats['spx_vol_5d'] = spx_ret.rolling(5, min_periods=3).std()
    feats['spx_abs_ret_max_5d'] = spx_ret.abs().rolling(5, min_periods=3).max()
    feats['spx_momentum_3d'] = spx.pct_change(3)
    feats['vix_spx_corr_30d'] = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    for w in [5, 10, 20]:
        ma = vix.rolling(w, min_periods=max(2, w // 2)).mean()
        feats[f'vix_vs_ma{w}'] = (vix - ma) / ma.replace(0, np.nan)
        feats[f'vix_zscore_{w}d'] = (vix - ma) / vix.rolling(w, min_periods=max(2, w // 2)).std().replace(0, np.nan)

    for h in [1, 2, 3, 5, 10, 20]:
        feats[f'vix_ret_{h}d'] = vix.pct_change(h)
        feats[f'spx_ret_{h}d'] = spx.pct_change(h)

    def rsi(series, n=14):
        delta = series.diff()
        gain = delta.clip(lower=0).rolling(n).mean()
        loss = (-delta.clip(upper=0)).rolling(n).mean()
        rs = gain / loss.replace(0, np.nan)
        return 100 - 100 / (1 + rs)

    feats['vix_rsi_14'] = rsi(vix, 14)
    feats['spx_rsi_14'] = rsi(spx, 14)

    for w in [10, 20]:
        ma = vix.rolling(w).mean()
        std = vix.rolling(w).std()
        feats[f'boll_width_{w}d'] = (2 * std) / ma.replace(0, np.nan)

    ema12 = vix.ewm(span=12).mean()
    ema26 = vix.ewm(span=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9).mean()
    feats['vix_macd'] = macd
    feats['vix_macd_signal'] = signal
    feats['vix_macd_hist'] = macd - signal
    feats['vix_roll_skew_20d'] = vix_ret.rolling(20).skew()
    feats['vix_roll_kurt_20d'] = vix_ret.rolling(20).kurt()

    for w in [20, 60]:
        roll_max = vix.rolling(w).max()
        roll_min = vix.rolling(w).min()
        feats[f'vix_dist_max_{w}d'] = (vix - roll_max) / roll_max.replace(0, np.nan)
        feats[f'vix_dist_min_{w}d'] = (vix - roll_min) / roll_min.replace(0, np.nan)

    roll_high = spx.rolling(252, min_periods=126).max()
    feats['spx_drawdown_252d'] = (spx - roll_high) / roll_high.replace(0, np.nan)

    # Z-score sur train uniquement.
    original_cols = list(feats.columns)
    for col in original_cols:
        mu = feats[col].iloc[:train_end_idx].mean()
        sd = feats[col].iloc[:train_end_idx].std()
        if np.isfinite(sd) and sd > 1e-8:
            feats[f'{col}_z'] = (feats[col] - mu) / sd

    return feats.replace([np.inf, -np.inf], np.nan)


In [7]:
# ============================================================
# 6. Interactions sur toutes les features candidates + sélection finale
# ============================================================

def _make_interaction_series(df, fi, fj, op, rolling_w=20, eps=1e-8):
    si = df[fi]
    sj = df[fj]
    if op == 'div':
        return si / sj.where(sj.abs() >= eps, np.nan)
    if op == 'minus':
        return si - sj
    if op == 'prod':
        return si * sj
    if op == 'zrel':
        diff = si - sj
        rs = diff.rolling(rolling_w, min_periods=max(2, rolling_w // 2)).std()
        return diff / rs.replace(0, np.nan)
    if op == 'macross':
        mai = si.rolling(rolling_w, min_periods=max(2, rolling_w // 2)).mean()
        maj = sj.rolling(rolling_w, min_periods=max(2, rolling_w // 2)).mean()
        return mai / maj.where(maj.abs() >= eps, np.nan)
    if op == 'ret5x':
        return si.pct_change(5) * sj
    raise ValueError(f'Opération inconnue : {op}')


def apply_interaction_specs(df, specs, rolling_w=20, eps=1e-8):
    cols = {}
    for fi, fj, op, name in specs:
        if fi not in df.columns or fj not in df.columns:
            continue
        try:
            cols[name] = _make_interaction_series(df, fi, fj, op, rolling_w=rolling_w, eps=eps)
        except Exception:
            pass
    if not cols:
        return pd.DataFrame(index=df.index)
    out = pd.DataFrame(cols, index=df.index).replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
    return out


def generate_interactions_all_features_batched(
    df,
    y,
    feature_cols,
    max_keep=3000,
    batch_size=1500,
    keep_per_batch=500,
    rolling_w=20,
    eps=1e-8,
    random_state=SEED
):
    feature_cols = [c for c in feature_cols if c in df.columns]
    ops = ['div', 'minus', 'prod', 'zrel', 'macross', 'ret5x']
    selected_chunks = []
    selected_specs = []
    selected_scores = []
    batch_cols = {}
    batch_specs = []
    y_arr = np.asarray(y).astype(int)

    total_pairs = len(feature_cols) * (len(feature_cols) - 1) // 2
    total_raw = total_pairs * len(ops)
    print(f"  [INTER ALL] {len(feature_cols)} features candidates")
    print(f"  [INTER ALL] Interactions théoriques : {total_raw}")

    def flush_batch(batch_cols, batch_specs):
        if not batch_cols:
            return None, [], pd.Series(dtype=float)
        tmp = pd.DataFrame(batch_cols, index=df.index)
        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')
        if tmp.shape[1] == 0:
            return None, [], pd.Series(dtype=float)
        tmp_clean = tmp.fillna(0)
        vals = np.nan_to_num(tmp_clean.values, nan=0.0, posinf=0.0, neginf=0.0)
        try:
            mi = mutual_info_classif(vals, y_arr, random_state=random_state)
            scores = pd.Series(mi, index=tmp_clean.columns).sort_values(ascending=False)
        except Exception:
            scores = tmp_clean.var().sort_values(ascending=False)
        keep_cols = scores.head(min(keep_per_batch, len(scores))).index.tolist()
        kept_specs = [spec for spec in batch_specs if spec[3] in keep_cols]
        return tmp_clean[keep_cols], kept_specs, scores.loc[keep_cols]

    counter = 0
    for i in range(len(feature_cols)):
        fi = feature_cols[i]
        for j in range(i + 1, len(feature_cols)):
            fj = feature_cols[j]
            for op in ops:
                name = f"{fi}__{op}__{fj}"
                try:
                    batch_cols[name] = _make_interaction_series(df, fi, fj, op, rolling_w=rolling_w, eps=eps)
                    batch_specs.append((fi, fj, op, name))
                    counter += 1
                except Exception:
                    continue
                if len(batch_cols) >= batch_size:
                    kept_df, kept_specs, kept_scores = flush_batch(batch_cols, batch_specs)
                    if kept_df is not None:
                        selected_chunks.append(kept_df)
                        selected_specs.extend(kept_specs)
                        selected_scores.append(kept_scores)
                    batch_cols = {}
                    batch_specs = []
                    print(f"    [INTER ALL] {counter}/{total_raw} interactions traitées")

    kept_df, kept_specs, kept_scores = flush_batch(batch_cols, batch_specs)
    if kept_df is not None:
        selected_chunks.append(kept_df)
        selected_specs.extend(kept_specs)
        selected_scores.append(kept_scores)

    if not selected_chunks:
        return pd.DataFrame(index=df.index), [], pd.Series(dtype=float)

    interactions = pd.concat(selected_chunks, axis=1)
    interactions = interactions.loc[:, ~interactions.columns.duplicated()]
    interactions = interactions.replace([np.inf, -np.inf], np.nan).fillna(0)

    all_scores = pd.concat(selected_scores) if selected_scores else pd.Series(dtype=float)
    all_scores = all_scores[~all_scores.index.duplicated(keep='first')].sort_values(ascending=False)

    if max_keep is not None and interactions.shape[1] > max_keep:
        final_cols = [c for c in all_scores.head(max_keep).index.tolist() if c in interactions.columns]
        interactions = interactions[final_cols]
        selected_specs = [spec for spec in selected_specs if spec[3] in final_cols]
        all_scores = all_scores.loc[final_cols]

    print(f"  [INTER ALL] Interactions retenues : {interactions.shape[1]}")
    return interactions, selected_specs, all_scores


def shap_select_features(X_train, y_train, top_n, label=''):
    try:
        from xgboost import XGBClassifier
        from sklearn.preprocessing import LabelEncoder
        X_clean = X_train.copy().replace([np.inf, -np.inf], np.nan).fillna(0)
        bad_cols = [c for c in X_clean.columns if not np.isfinite(X_clean[c].values).all()]
        if bad_cols:
            X_clean = X_clean.drop(columns=bad_cols)
        le = LabelEncoder()
        y_enc = le.fit_transform(y_train)
        pilot = XGBClassifier(
            n_estimators=80,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            eval_metric='mlogloss',
            objective='multi:softprob',
            random_state=SEED,
            n_jobs=-1
        )
        pilot.fit(X_clean.values, y_enc)
        expl = shap.TreeExplainer(pilot)
        sv = expl.shap_values(X_clean.values[:500])
        if isinstance(sv, list):
            arr = np.mean([np.abs(s) for s in sv], axis=0)
        elif np.array(sv).ndim == 3:
            arr = np.abs(sv).mean(axis=2)
        else:
            arr = np.abs(sv)
        scores = pd.Series(arr.mean(axis=0), index=X_clean.columns)
        top = scores.sort_values(ascending=False).head(top_n).index.tolist()
        if label:
            print(f"  [SHAP {label}] top-{len(top)}/{len(scores)}")
        return top, scores.sort_values(ascending=False)
    except Exception as e:
        print(f"  [WARN] SHAP indisponible, fallback mutual information: {e}")
        X_clean = X_train.copy().replace([np.inf, -np.inf], np.nan).fillna(0)
        vals = np.nan_to_num(X_clean.values, nan=0.0, posinf=0.0, neginf=0.0)
        mi = mutual_info_classif(vals, y_train, random_state=SEED)
        scores = pd.Series(mi, index=X_clean.columns).sort_values(ascending=False)
        return scores.head(top_n).index.tolist(), scores


In [8]:
# ============================================================
# 7. Dataset PyTorch
# ============================================================

class VIXAmplitudeDataset(Dataset):
    def __init__(self, data, feature_cols, target_col=TARGET_COL, lookback=CONFIG['lookback'], scaler=None):
        self.lookback = lookback
        X = data[feature_cols].copy().replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
        y = data[target_col].values.astype(int)
        self.scaler = scaler if scaler is not None else RobustScaler()
        self.X = self.scaler.fit_transform(X) if scaler is None else self.scaler.transform(X)
        self.X = np.nan_to_num(self.X, nan=0.0, posinf=0.0, neginf=0.0)
        self.y = y
        self.n_features = self.X.shape[1]

    def __len__(self):
        return max(0, len(self.X) - self.lookback)

    def __getitem__(self, idx):
        x_seq = torch.tensor(self.X[idx:idx + self.lookback], dtype=torch.float32)
        y_val = torch.tensor(self.y[idx + self.lookback], dtype=torch.long)
        return x_seq, y_val


In [9]:
# ============================================================
# 8. Architectures Deep Learning
# ============================================================

class VIX_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, n_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.norm(out[:, -1, :]))

class TCNBlock(nn.Module):
    def __init__(self, n_in, n_out, kernel_size, dilation, dropout=0.2):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.conv = nn.utils.weight_norm(nn.Conv1d(n_in, n_out, kernel_size, padding=pad, dilation=dilation))
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.act = nn.GELU()
    def forward(self, x):
        conv_out = self.conv(x)
        if self.conv.padding[0] > 0:
            conv_out = conv_out[:, :, :-self.conv.padding[0]]
        out = self.act(self.drop(conv_out))
        res = x if self.skip is None else self.skip(x)
        return self.act(out + res)

class VIX_TCN(nn.Module):
    def __init__(self, input_dim, channels=[32, 64, 128], kernel_size=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        layers = []
        in_ch = input_dim
        for i, ch in enumerate(channels):
            layers.append(TCNBlock(in_ch, ch, kernel_size, dilation=2 ** i, dropout=dropout))
            in_ch = ch
        self.net = nn.Sequential(*layers)
        self.fc = nn.Sequential(nn.Linear(channels[-1], 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.net(x)
        return self.fc(x.mean(dim=2))

class VIX_Transformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos_emb = nn.Embedding(CONFIG['lookback'], d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        B, T, _ = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = self.proj(x) + self.pos_emb(pos)
        out = self.encoder(x)
        return self.fc(out[:, -1, :])

class VIX_CNNLSTM(nn.Module):
    def __init__(self, input_dim, conv_filters=64, lstm_hidden=128, kernel_size=3, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, conv_filters, kernel_size=kernel_size, padding='same'),
            nn.GELU(), nn.BatchNorm1d(conv_filters), nn.Dropout(dropout)
        )
        self.lstm = nn.LSTM(conv_filters, lstm_hidden, batch_first=True, num_layers=2, dropout=dropout)
        self.fc = nn.Sequential(nn.Linear(lstm_hidden, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        x = self.conv(x.transpose(1, 2)).transpose(1, 2)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class NBeatsBlock(nn.Module):
    def __init__(self, input_size, theta_size, hidden_size=256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, theta_size)
        )
    def forward(self, x):
        return self.fc(x)

class VIX_NBeats(nn.Module):
    def __init__(self, input_dim, lookback, n_blocks=3, hidden_size=256, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.blocks = nn.ModuleList([NBeatsBlock(input_dim * lookback, hidden_size, hidden_size) for _ in range(n_blocks)])
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        residual = x.reshape(x.size(0), -1)
        out = None
        for block in self.blocks:
            theta = self.drop(block(residual))
            out = theta if out is None else out + theta
        return self.fc(out)

class GatedLinearUnit(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.gate = nn.Linear(in_dim, out_dim)
    def forward(self, x):
        return self.fc(x) * torch.sigmoid(self.gate(x))

class VIX_TFT(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, lstm_layers=2, dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.vsn = nn.Sequential(nn.Linear(input_dim, d_model), nn.GELU(), nn.Linear(d_model, input_dim), nn.Softmax(dim=-1))
        self.input_proj = nn.Linear(input_dim, d_model)
        self.lstm = nn.LSTM(d_model, d_model, num_layers=lstm_layers, batch_first=True, dropout=dropout)
        self.lstm_norm = nn.LayerNorm(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_model * 2, dropout=dropout, batch_first=True, norm_first=True)
        self.attention = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.glu = GatedLinearUnit(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        weights = self.vsn(x.mean(dim=1, keepdim=True))
        x = x * weights
        x = self.input_proj(x)
        lstm_out, _ = self.lstm(x)
        lstm_out = self.lstm_norm(lstm_out + x)
        attn_out = self.attention(lstm_out)
        out = self.norm(self.glu(attn_out) + attn_out)
        return self.fc(out[:, -1, :])

print('Architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT')


Architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT


In [10]:
# ============================================================
# 9. Entraînement et métriques complètes
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ls = label_smoothing
        self.reduction = reduction
    def forward(self, logits, targets):
        n_cls = logits.size(1)
        one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1)
        smooth = one_hot * (1 - self.ls) + self.ls / n_cls
        log_prob = torch.log_softmax(logits, dim=1)
        prob = log_prob.exp()
        alpha_t = self.alpha.to(logits.device)[targets] if self.alpha is not None else 1.0
        focal_weight = (1 - prob) ** self.gamma
        loss = -(alpha_t.unsqueeze(1) * focal_weight * smooth * log_prob).sum(dim=1)
        return loss.mean() if self.reduction == 'mean' else loss.sum()


def compute_class_weights(y_train, n_classes=4):
    counts = np.bincount(y_train, minlength=n_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * n_classes
    return torch.tensor(weights, dtype=torch.float32)


def train_model(model, train_loader, val_loader, class_weights=None, epochs=CONFIG['epochs'], lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'], label='Modèle', patience=10):
    criterion = FocalLoss(gamma=2.0, alpha=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    best_val_loss = float('inf')
    best_state = None
    wait = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        model.eval()
        val_loss = 0.0
        val_preds, val_targets = [], []
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device)
                logits = model(bx)
                val_loss += criterion(logits, by).item()
                val_preds.extend(logits.argmax(1).cpu().numpy())
                val_targets.extend(by.cpu().numpy())
        val_loss_avg = val_loss / max(1, len(val_loader))
        train_loss_avg = train_loss / max(1, len(train_loader))
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0) if len(val_targets) else 0
        history['train_loss'].append(train_loss_avg)
        history['val_loss'].append(val_loss_avg)
        history['val_f1'].append(val_f1)

        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  [{label}] Early stop @ epoch {epoch + 1} ({time.time() - t0:.1f}s)")
                break
        if (epoch + 1) % 10 == 0:
            print(f"  [{label}] ep {epoch + 1}/{epochs} | val_f1={val_f1:.4f} | {time.time() - t0:.1f}s")

    if best_state:
        model.load_state_dict(best_state)
    return history


def safe_auc_multiclass(y_true, y_prob, n_classes=4):
    try:
        if y_prob.ndim != 2 or y_prob.shape[1] != n_classes or len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro', labels=list(range(n_classes)))
    except Exception:
        return np.nan


def safe_auc_binary(y_true_binary, y_score):
    try:
        if len(np.unique(y_true_binary)) < 2:
            return np.nan
        return roc_auc_score(y_true_binary, y_score)
    except Exception:
        return np.nan


def compute_metrics_from_arrays(y_true, y_pred, y_prob, label=''):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob)

    metrics = {
        'Acc_4cls': accuracy_score(y_true, y_pred),
        'Precision_4cls_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'Precision_4cls_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall_4cls_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'Recall_4cls_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1_4cls_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'F1_4cls_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'AUC_4cls_ovr_macro': safe_auc_multiclass(y_true, y_prob, n_classes=4),
    }
    metrics['F1_4cls'] = metrics['F1_4cls_macro']

    y_true_dir = np.where(np.isin(y_true, [2, 3]), 1, 0)
    y_pred_dir = np.where(np.isin(y_pred, [2, 3]), 1, 0)
    y_prob_up = y_prob[:, 2] + y_prob[:, 3] if y_prob.ndim == 2 and y_prob.shape[1] >= 4 else np.full(len(y_true), np.nan)

    metrics.update({
        'Acc_dir': accuracy_score(y_true_dir, y_pred_dir),
        'Precision_dir_macro': precision_score(y_true_dir, y_pred_dir, average='macro', zero_division=0),
        'Precision_UP': precision_score(y_true_dir, y_pred_dir, pos_label=1, average='binary', zero_division=0),
        'Precision_DOWN': precision_score(y_true_dir, y_pred_dir, pos_label=0, average='binary', zero_division=0),
        'Recall_dir_macro': recall_score(y_true_dir, y_pred_dir, average='macro', zero_division=0),
        'Recall_UP': recall_score(y_true_dir, y_pred_dir, pos_label=1, average='binary', zero_division=0),
        'Recall_DOWN': recall_score(y_true_dir, y_pred_dir, pos_label=0, average='binary', zero_division=0),
        'F1_dir': f1_score(y_true_dir, y_pred_dir, average='macro', zero_division=0),
        'F1_UP': f1_score(y_true_dir, y_pred_dir, pos_label=1, average='binary', zero_division=0),
        'F1_DOWN': f1_score(y_true_dir, y_pred_dir, pos_label=0, average='binary', zero_division=0),
        'AUC_dir': safe_auc_binary(y_true_dir, y_prob_up),
    })

    up_idx = np.where(np.isin(y_true, [2, 3]))[0]
    dn_idx = np.where(np.isin(y_true, [0, 1]))[0]
    if len(up_idx) > 0:
        yt_up = np.where(y_true[up_idx] == 3, 1, 0)
        yp_up = np.where(y_pred[up_idx] == 3, 1, 0)
        metrics['Acc_UP_sub'] = accuracy_score(yt_up, yp_up)
        metrics['Precision_UP_FORT'] = precision_score(yt_up, yp_up, zero_division=0)
        metrics['Recall_UP_FORT'] = recall_score(yt_up, yp_up, zero_division=0)
        metrics['F1_UP_FORT'] = f1_score(yt_up, yp_up, zero_division=0)
        if y_prob.ndim == 2 and y_prob.shape[1] >= 4:
            prob_up_fort_cond = y_prob[up_idx, 3] / np.clip(y_prob[up_idx, 2] + y_prob[up_idx, 3], 1e-8, None)
            metrics['AUC_UP_FORT'] = safe_auc_binary(yt_up, prob_up_fort_cond)
    if len(dn_idx) > 0:
        yt_dn = np.where(y_true[dn_idx] == 0, 1, 0)
        yp_dn = np.where(y_pred[dn_idx] == 0, 1, 0)
        metrics['Acc_DOWN_sub'] = accuracy_score(yt_dn, yp_dn)
        metrics['Precision_DOWN_FORT'] = precision_score(yt_dn, yp_dn, zero_division=0)
        metrics['Recall_DOWN_FORT'] = recall_score(yt_dn, yp_dn, zero_division=0)
        metrics['F1_DOWN_FORT'] = f1_score(yt_dn, yp_dn, zero_division=0)
        if y_prob.ndim == 2 and y_prob.shape[1] >= 4:
            prob_down_fort_cond = y_prob[dn_idx, 0] / np.clip(y_prob[dn_idx, 0] + y_prob[dn_idx, 1], 1e-8, None)
            metrics['AUC_DOWN_FORT'] = safe_auc_binary(yt_dn, prob_down_fort_cond)

    if label:
        print(f"  [{label}] Acc_dir={metrics['Acc_dir']:.4f} | Precision_dir={metrics['Precision_dir_macro']:.4f} | Recall_dir={metrics['Recall_dir_macro']:.4f} | F1_dir={metrics['F1_dir']:.4f} | AUC_dir={metrics['AUC_dir'] if np.isfinite(metrics['AUC_dir']) else np.nan:.4f}")
    return metrics


def evaluate_model(model, loader, label=''):
    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    with torch.no_grad():
        for bx, by in loader:
            logits = model(bx.to(device))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_targets.extend(by.numpy())
    y_true = np.array(all_targets).astype(int)
    y_pred = np.array(all_preds).astype(int)
    y_prob = np.array(all_probs)
    metrics = compute_metrics_from_arrays(y_true, y_pred, y_prob, label=label)
    return metrics, y_pred, y_prob, y_true


In [11]:
# ============================================================
# 10. Graphes de visualisation
# ============================================================

def plot_training_histories(histories, horizon=None):
    if not histories:
        print('[PLOT] Aucun historique disponible.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for name, hist in histories.items():
        if 'train_loss' in hist: axes[0].plot(hist['train_loss'], label=f'{name} train')
        if 'val_loss' in hist: axes[0].plot(hist['val_loss'], linestyle='--', label=f'{name} val')
        if 'val_f1' in hist: axes[1].plot(hist['val_f1'], label=name)
    suffix = f' — h={horizon}j' if horizon is not None else ''
    axes[0].set_title(f"Loss entraînement / validation{suffix}")
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=8)
    axes[1].set_title(f"F1 validation macro{suffix}")
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 macro')
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix_4cls(y_true, y_pred, title='Matrice de confusion 4 classes'):
    labels = CONFIG.get('class_labels', ['DOWN_FORT', 'DOWN_FAIBLE', 'UP_FAIBLE', 'UP_FORT'])
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel('Prédiction')
    plt.ylabel('Vraie classe')
    plt.tight_layout()
    plt.show()


def plot_roc_direction(y_true, y_prob, title='Courbe ROC directionnelle UP vs DOWN'):
    if y_prob.ndim != 2 or y_prob.shape[1] < 4:
        print('[PLOT] Probabilités invalides pour ROC.')
        return
    y_true_dir = np.where(np.isin(y_true, [2, 3]), 1, 0)
    y_score_up = y_prob[:, 2] + y_prob[:, 3]
    if len(np.unique(y_true_dir)) < 2:
        print('[PLOT] Une seule classe directionnelle présente.')
        return
    fpr, tpr, _ = roc_curve(y_true_dir, y_score_up)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.title(title)
    plt.xlabel('Taux de faux positifs')
    plt.ylabel('Taux de vrais positifs')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_feature_scores(scores, top_n=30, title='Importance des features'):
    if scores is None or len(scores) == 0:
        print('[PLOT] Aucun score disponible.')
        return
    top = scores.sort_values(ascending=False).head(top_n)
    plt.figure(figsize=(10, 8))
    top.sort_values().plot(kind='barh')
    plt.title(title)
    plt.xlabel('Score')
    plt.tight_layout()
    plt.show()


def plot_metrics_bar(df_report, horizon=None):
    df_plot = df_report.copy()
    if horizon is not None:
        df_plot = df_plot[df_plot['Horizon'] == horizon].copy()
    metric_cols = ['Acc_dir', 'Precision_dir_macro', 'Recall_dir_macro', 'F1_dir', 'AUC_dir', 'Acc_4cls', 'F1_4cls_macro', 'AUC_4cls_ovr_macro']
    metric_cols = [c for c in metric_cols if c in df_plot.columns]
    if df_plot.empty or not metric_cols:
        print('[PLOT] Pas assez de données.')
        return
    df_long = df_plot.melt(id_vars=['Horizon', 'Modèle'], value_vars=metric_cols, var_name='Métrique', value_name='Valeur')
    plt.figure(figsize=(16, 7))
    sns.barplot(data=df_long, x='Modèle', y='Valeur', hue='Métrique')
    plt.title('Comparaison des métriques par modèle' + (f' — h={horizon}j' if horizon else ''))
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.show()


def plot_metrics_heatmap(df_report):
    if df_report.empty:
        print('[PLOT] Rapport vide.')
        return
    metrics = ['F1_dir', 'AUC_dir', 'Acc_dir', 'F1_4cls_macro', 'AUC_4cls_ovr_macro']
    metrics = [m for m in metrics if m in df_report.columns]
    for metric in metrics:
        pivot = df_report.pivot_table(index='Modèle', columns='Horizon', values=metric, aggfunc='mean')
        plt.figure(figsize=(9, 5))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1)
        plt.title(f'Heatmap — {metric}')
        plt.tight_layout()
        plt.show()


In [12]:
import torch
import numpy as np
import pandas as pd
import time
from sklearn.metrics import f1_score
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn

from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek

def select_best_sampler(X_train, y_train):
    from sklearn.ensemble import RandomForestClassifier
    samplers = {
        'SMOTE': SMOTE(random_state=SEED),
        'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
        'SMOTETomek': SMOTETomek(random_state=SEED),
    }
    pilot = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=SEED, n_jobs=-1)
    best_name, best_f1, best_sampler = 'SMOTE', -1, samplers['SMOTE']
    for name, samp in samplers.items():
        try:
            Xr, yr = samp.fit_resample(X_train, y_train)
            n = len(Xr)
            scores = []
            for fold in range(3):
                t_end = n * (fold + 2) // 4
                v_end = n * (fold + 3) // 4
                pilot.fit(Xr[:t_end], yr[:t_end])
                preds = pilot.predict(Xr[t_end:v_end])
                scores.append(f1_score(yr[t_end:v_end], preds, average='macro', zero_division=0))
            avg = np.mean(scores)
            if avg > best_f1:
                best_f1, best_name, best_sampler = avg, name, samp
        except Exception:
            pass
    return best_name, best_sampler


def run_full_pipeline(df_raw, horizon=5, make_plots=True):
    t_total = time.time()
    print(f"{'=' * 60}PIPELINE h={horizon}j{'=' * 60}")

    all_dates = df_raw.dropna(how='all').index.sort_values()
    split_idx = int(len(all_dates) * 0.80)
    split_date = all_dates[split_idx]
    print(f"  Split 80/20 : train -> {all_dates[split_idx - 1].date()} | test -> {split_date.date()}")

    ts_feats = build_ts_features(df_raw, split_idx)
    vix_col = 'IDX_VIX' if 'IDX_VIX' in df_raw.columns else [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c][0]
    spx_col = [c for c in df_raw.columns if 'GSPC' in c or 'SPY' in c][0]
    adv_feats = build_advanced_features(df_raw, df_raw[vix_col], df_raw[spx_col], split_idx)

    ret_feats = []
    for col in df_raw.columns:
        for w in [1, 5, 21]:
            s = df_raw[col].pct_change(w)
            s.name = f'{col}_ret{w}d'
            ret_feats.append(s)
    ret_df = pd.concat(ret_feats, axis=1)

    df_all = pd.concat([df_raw, ts_feats, adv_feats, ret_df], axis=1).replace([np.inf, -np.inf], np.nan)
    target, regime, vix_ret_target, thresholds = build_amplitude_target(df_raw[vix_col], horizon, split_idx)
    df_all = df_all.reindex(target.index)
    df_all[TARGET_COL] = target

    feature_cols = [c for c in df_all.columns if c != TARGET_COL]
    df_train = df_all.loc[df_all.index < split_date].dropna(subset=[TARGET_COL])
    df_test = df_all.loc[df_all.index >= split_date].dropna(subset=[TARGET_COL])
    y_tr = df_train[TARGET_COL].values.astype(int)
    y_te = df_test[TARGET_COL].values.astype(int)

    X_tr_base = df_train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_te_base = df_test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    print(f"  Base features avant interactions : {len(feature_cols)}")
    print('  Engineering interactions sur toutes les features candidates...')
    idf_train, interaction_specs, interaction_scores = generate_interactions_all_features_batched(
        X_tr_base,
        y_tr,
        feature_cols=feature_cols,
        max_keep=CONFIG.get('max_interaction_keep', 3000),
        batch_size=CONFIG.get('interaction_batch_size', 1500),
        keep_per_batch=CONFIG.get('interaction_keep_per_batch', 500),
        rolling_w=20,
        eps=1e-8
    )
    idf_test = apply_interaction_specs(X_te_base, interaction_specs, rolling_w=20, eps=1e-8)
    interaction_cols = idf_train.columns.tolist()
    for col in interaction_cols:
        if col not in idf_test.columns:
            idf_test[col] = 0.0
    idf_test = idf_test[interaction_cols]

    df_train_ext = pd.concat([X_tr_base, idf_train], axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)
    df_test_ext = pd.concat([X_te_base, idf_test], axis=1).replace([np.inf, -np.inf], np.nan).fillna(0)
    ext_cols = df_train_ext.columns.tolist()
    print(f"  Features étendues avant sélection finale : {len(ext_cols)}")

    sc_ext = RobustScaler()
    X_tr_ext = pd.DataFrame(sc_ext.fit_transform(df_train_ext), columns=ext_cols, index=df_train.index).replace([np.inf, -np.inf], np.nan).fillna(0)
    X_te_ext = pd.DataFrame(sc_ext.transform(df_test_ext), columns=ext_cols, index=df_test.index).replace([np.inf, -np.inf], np.nan).fillna(0)

    top_final, scores_final = shap_select_features(X_tr_ext, y_tr, CONFIG['top_n_final'], label='final_all_features')
    print(f"  Features finales : {len(top_final)}")
    if make_plots:
        plot_feature_scores(scores_final, top_n=min(30, len(scores_final)), title=f'Top features sélectionnées — h={horizon}j')

    X_final_tr = X_tr_ext[top_final].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_final_te = X_te_ext[top_final].replace([np.inf, -np.inf], np.nan).fillna(0)

    sampler_name, best_sampler = select_best_sampler(X_final_tr.values, y_tr)
    print(f"  Sampler sélectionné : {sampler_name}")
    X_res, y_res = best_sampler.fit_resample(X_final_tr.values, y_tr)

    df_res = pd.DataFrame(X_res, columns=top_final)
    df_res[TARGET_COL] = y_res
    df_te_final = pd.DataFrame(X_final_te.values, columns=top_final, index=df_test.index)
    df_te_final[TARGET_COL] = y_te

    val_split = int(len(df_res) * 0.85)
    sc_seq = RobustScaler().fit(df_res[top_final].iloc[:val_split])
    ds_train = VIXAmplitudeDataset(df_res.iloc[:val_split], top_final, scaler=sc_seq)
    ds_val = VIXAmplitudeDataset(df_res.iloc[val_split:], top_final, scaler=sc_seq)
    ds_test = VIXAmplitudeDataset(df_te_final, top_final, scaler=sc_seq)
    dl_train = DataLoader(ds_train, batch_size=CONFIG['batch_size'], shuffle=True)
    dl_val = DataLoader(ds_val, batch_size=256)
    dl_test = DataLoader(ds_test, batch_size=256)

    input_dim = len(top_final)
    class_weights = compute_class_weights(y_res)
    models_def = {
        'LSTM': VIX_LSTM(input_dim),
        'TCN': VIX_TCN(input_dim),
        'Transformer': VIX_Transformer(input_dim),
        'CNN-LSTM': VIX_CNNLSTM(input_dim),
        'N-BEATS': VIX_NBeats(input_dim, CONFIG['lookback']),
        'TFT': VIX_TFT(input_dim),
    }

    results_all, trained_models, histories, predictions = {}, {}, {}, {}
    for name, model in models_def.items():
        print(f"\n  -- {name} --")
        model = model.to(device)
        hist = train_model(model, dl_train, dl_val, class_weights=class_weights, label=name)
        met, preds, probs, y_true_eval = evaluate_model(model, dl_test, label=name)
        results_all[name] = met
        trained_models[name] = model
        histories[name] = hist
        predictions[name] = {'y_true': y_true_eval, 'y_pred': preds, 'y_prob': probs}

    # Ensemble pondéré par F1 directionnel.
    model_names = list(models_def.keys())
    weights = np.array([results_all[n].get('F1_dir', 0.0) if np.isfinite(results_all[n].get('F1_dir', 0.0)) else 0.0 for n in model_names])
    weights = weights - weights.min() + 1e-6
    weights = np.ones_like(weights) / len(weights) if weights.sum() <= 0 else weights / weights.sum()
    all_probs_list = [predictions[n]['y_prob'] for n in model_names]
    min_len = min(len(p) for p in all_probs_list)
    all_probs_list = [p[:min_len] for p in all_probs_list]
    y_true_ens = predictions[model_names[0]]['y_true'][:min_len]
    ensemble_probs = np.sum([p * w for p, w in zip(all_probs_list, weights)], axis=0)
    ensemble_preds = ensemble_probs.argmax(axis=1)
    ensemble_metrics = compute_metrics_from_arrays(y_true_ens, ensemble_preds, ensemble_probs, label='ENSEMBLE')
    results_all['ENSEMBLE'] = ensemble_metrics
    predictions['ENSEMBLE'] = {'y_true': y_true_ens, 'y_pred': ensemble_preds, 'y_prob': ensemble_probs}

    print(f"\n  Ensemble F1_dir = {ensemble_metrics['F1_dir']:.4f} (poids: {dict(zip(model_names, weights.round(3)))})")

    if make_plots:
        plot_training_histories(histories, horizon=horizon)
        best_name = max(results_all, key=lambda n: results_all[n].get('F1_dir', -np.inf))
        print(f"  Meilleur modèle h={horizon}j selon F1_dir : {best_name}")
        plot_confusion_matrix_4cls(predictions[best_name]['y_true'], predictions[best_name]['y_pred'], title=f'Matrice de confusion — {best_name} — h={horizon}j')
        plot_roc_direction(predictions[best_name]['y_true'], predictions[best_name]['y_prob'], title=f'ROC directionnelle — {best_name} — h={horizon}j')

    print(f"\n  Pipeline h={horizon}j terminé ({time.time() - t_total:.1f}s)")
    return {
        'models': trained_models,
        'results': results_all,
        'ensemble_f1': ensemble_metrics['F1_dir'],
        'ensemble_metrics': ensemble_metrics,
        'features': top_final,
        'interaction_specs': interaction_specs,
        'feature_scores': scores_final,
        'interaction_scores': interaction_scores,
        'scaler': sc_seq,
        'histories': histories,
        'predictions': predictions,
    }

In [ ]:
# ============================================================
# 12. Exécution de tous les horizons
# ============================================================

all_results = {}

for h in CONFIG['horizons']:
    all_results[h] = run_full_pipeline(df_raw, horizon=h, make_plots=True)

print('' + '=' * 80)
print('RÉSUMÉ — MEILLEUR MODÈLE ET ENSEMBLE PAR HORIZON')
print('=' * 80)
for h, res in all_results.items():
    best_indiv = max(
        [(k, v) for k, v in res['results'].items() if k != 'ENSEMBLE'],
        key=lambda x: x[1].get('F1_dir', -np.inf)
    )
    print(f"h={h}j | Ensemble F1_dir={res['ensemble_f1']:.4f} | Best individuel: {best_indiv[0]} ({best_indiv[1].get('F1_dir', np.nan):.4f})")


============================================================PIPELINE h=1j============================================================
  Split 80/20 : train -> 2023-08-18 | test -> 2023-08-21
  EGARCH fit OK (0.1s)
  Kalman fit OK (56.1s)
  HMM fit OK (56.2s)
  TS features OK (61.2s)
  [Target h=1j] 3500 obs, dist: {1: 1046, 3: 875, 0: 860, 2: 719}
  Base features avant interactions : 373
  Engineering interactions sur toutes les features candidates...
  [INTER ALL] 373 features candidates
  [INTER ALL] Interactions théoriques : 416268
    [INTER ALL] 1500/416268 interactions traitées
    [INTER ALL] 3000/416268 interactions traitées
    [INTER ALL] 4500/416268 interactions traitées
    [INTER ALL] 6000/416268 interactions traitées
    [INTER ALL] 7500/416268 interactions traitées
    [INTER ALL] 9000/416268 interactions traitées
    [INTER ALL] 10500/416268 interactions traitées
    [INTER ALL] 12000/416268 interactions traitées
    [INTER ALL] 13500/416268 interactions traitées
    [I

In [2]:
# ============================================================
# 13. Rapport final enrichi et export Excel
# ============================================================

def build_report(all_results):
    rows = []
    for h, res in all_results.items():
        for model_name, met in res['results'].items():
            rows.append({
                'Horizon': h,
                'Modèle': model_name,
                'Acc_dir': round(met.get('Acc_dir', np.nan), 4),
                'Precision_dir_macro': round(met.get('Precision_dir_macro', np.nan), 4),
                'Recall_dir_macro': round(met.get('Recall_dir_macro', np.nan), 4),
                'F1_dir': round(met.get('F1_dir', np.nan), 4),
                'AUC_dir': round(met.get('AUC_dir', np.nan), 4),
                'Acc_4cls': round(met.get('Acc_4cls', np.nan), 4),
                'Precision_4cls_macro': round(met.get('Precision_4cls_macro', np.nan), 4),
                'Recall_4cls_macro': round(met.get('Recall_4cls_macro', np.nan), 4),
                'F1_4cls_macro': round(met.get('F1_4cls_macro', np.nan), 4),
                'AUC_4cls_ovr_macro': round(met.get('AUC_4cls_ovr_macro', np.nan), 4),
                'F1_UP': round(met.get('F1_UP', np.nan), 4),
                'F1_DOWN': round(met.get('F1_DOWN', np.nan), 4),
                'F1_UP_FORT': round(met.get('F1_UP_FORT', np.nan), 4),
                'F1_DOWN_FORT': round(met.get('F1_DOWN_FORT', np.nan), 4),
                'Precision_UP_FORT': round(met.get('Precision_UP_FORT', np.nan), 4),
                'Recall_UP_FORT': round(met.get('Recall_UP_FORT', np.nan), 4),
                'AUC_UP_FORT': round(met.get('AUC_UP_FORT', np.nan), 4),
                'Precision_DOWN_FORT': round(met.get('Precision_DOWN_FORT', np.nan), 4),
                'Recall_DOWN_FORT': round(met.get('Recall_DOWN_FORT', np.nan), 4),
                'AUC_DOWN_FORT': round(met.get('AUC_DOWN_FORT', np.nan), 4),
            })
    return pd.DataFrame(rows)


df_report = build_report(all_results)
print('=' * 100)
print('RÉSULTATS DL — métriques complètes')
print('=' * 100)
print(df_report.to_string(index=False))

plot_metrics_heatmap(df_report)
for h in sorted(df_report['Horizon'].dropna().unique()):
    plot_metrics_bar(df_report, horizon=h)

try:
    with pd.ExcelWriter('vix_dl_report_enriched.xlsx', engine='xlsxwriter') as w:
        df_report.to_excel(w, sheet_name='DL_Results_Full', index=False)

        feature_rows = []
        for h, res in all_results.items():
            for rank, feat in enumerate(res.get('features', []), start=1):
                feature_rows.append({'Horizon': h, 'Rank': rank, 'Feature': feat})
        if feature_rows:
            pd.DataFrame(feature_rows).to_excel(w, sheet_name='Selected_Features', index=False)

        inter_rows = []
        for h, res in all_results.items():
            scores = res.get('interaction_scores', None)
            if scores is not None and len(scores) > 0:
                for rank, (feat, score) in enumerate(scores.head(500).items(), start=1):
                    inter_rows.append({'Horizon': h, 'Rank': rank, 'Interaction': feat, 'Score': score})
        if inter_rows:
            pd.DataFrame(inter_rows).to_excel(w, sheet_name='Top_Interactions', index=False)

    print('[SAVE] vix_dl_report_enriched.xlsx')
except Exception as e:
    print(f'[WARN] Export Excel: {e}')


NameError: name 'all_results' is not defined

## 14. Suggestions méthodologiques ajoutées

1. **Comparer systématiquement la version avec interactions et sans interactions.**  
   L'engineering sur toutes les features augmente fortement le risque d'overfitting. Il faut donc isoler l'apport réel des interactions.

2. **Remplacer l'ensemble pondéré par un méta-modèle.**  
   Si l'ensemble pondéré sous-performe le meilleur modèle individuel, une régression logistique ou un XGBoost sur les probabilités des modèles peut être plus robuste.

3. **Ajouter une validation par régime de marché.**  
   Les performances doivent être calculées séparément en régime CALM, NORMAL et STRESS afin de vérifier si le modèle est vraiment utile dans les périodes de stress.

4. **Ajouter une calibration des probabilités.**  
   Une bonne AUC ne signifie pas que les probabilités softmax sont utilisables directement. Un temperature scaling ou Platt scaling peut améliorer la qualité des probabilités.

5. **Ajouter une métrique économique.**  
   Pour le VIX, les métriques ML doivent être complétées par un backtest simple : signal UP/DOWN, coût de transaction fictif, hit ratio, rendement moyen par signal et drawdown.

## Glossaire

- **Accuracy** : proportion totale de prédictions correctes.
- **Precision** : proportion de prédictions positives réellement correctes.
- **Recall** : proportion des vrais positifs détectés par le modèle.
- **F1-score** : moyenne harmonique entre precision et recall.
- **AUC** : mesure de la capacité du modèle à classer correctement les probabilités.
- **Feature engineering** : création de variables explicatives à partir des données existantes.
- **Interaction de features** : combinaison de deux variables, par exemple ratio, différence ou produit.
- **SHAP** : méthode d'interprétabilité estimant la contribution de chaque variable aux prédictions.
- **SMOTE** : méthode de rééquilibrage des classes par génération d'observations synthétiques.
- **Overfitting** : situation où le modèle apprend trop bien l'échantillon d'entraînement et généralise mal.
